## Data Ingestion

In [1]:
  ###Document Structure
from langchain_core.documents import Document

In [2]:
doc=Document(
  page_content="This is a sample document.",
)

In [3]:
## simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)


In [4]:
# text loader
# from langchain_core.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/Rag.txt",encoding="utf-8")
document = loader.load()
print(document)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_12784\682465810.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\WEB DEV\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/Rag.txt'}, page_content='Retrieval-Augmented Generation, commonly known as RAG, is an AI architecture that combines information retrieval systems with large language models. The main goal of RAG is to improve the accuracy and reliability of AI-generated responses by providing external knowledge during generation.\n\nTraditional large language models rely only on the information stored during training. This means they may produce outdated or incorrect answers when asked about new or domain-specific topics. RAG solves this problem by retrieving relevant information from external sources before generating a response.\n\nA typical RAG pipeline contains several stages. First, documents are collected from sources such as PDFs, websites, databases, or text files. These documents are then split into smaller chunks to make retrieval more efficient. Each chunk is converted into vector embeddings using embedding models.\n\nThe embeddings are store

In [5]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_core import documents
dir_loader = DirectoryLoader(
  "../data/text_files", 
  glob="**/*.txt",
  loader_cls=TextLoader,
  loader_kwargs={"encoding": "utf-8"},
  show_progress=False)
txt_documents = dir_loader.load()
txt_documents

[Document(metadata={'source': '..\\data\\text_files\\Rag.txt'}, page_content='Retrieval-Augmented Generation, commonly known as RAG, is an AI architecture that combines information retrieval systems with large language models. The main goal of RAG is to improve the accuracy and reliability of AI-generated responses by providing external knowledge during generation.\n\nTraditional large language models rely only on the information stored during training. This means they may produce outdated or incorrect answers when asked about new or domain-specific topics. RAG solves this problem by retrieving relevant information from external sources before generating a response.\n\nA typical RAG pipeline contains several stages. First, documents are collected from sources such as PDFs, websites, databases, or text files. These documents are then split into smaller chunks to make retrieval more efficient. Each chunk is converted into vector embeddings using embedding models.\n\nThe embeddings are st

In [2]:
### pdf Loader
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
dir_loader = DirectoryLoader(
  "../data/pdfs", 
  glob="**/*.pdf",
  loader_cls=PyMuPDFLoader,
  show_progress=True)
pdf_documents = dir_loader.load()
pdf_documents

C:\Users\ASUS\AppData\Local\Temp\ipykernel_24740\174970157.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
d:\WEB DEV\SNBoseProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'DirectoryLoader' is not defined

In [7]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### embedding and cheunking

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity



d:\WEB DEV\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
class EmbeddingManager:
  def __init__(
    self,
    model_name: str = "all-MiniLM-L6-v2"):
    self.model_name = model_name
    self.model = None
    self._load_model()
  def _load_model(self):
    """Load the sentence transformer model."""
    try:
      print(f"Loading model: {self.model_name}...")
      self.model = SentenceTransformer(self.model_name)
      print(f"Model loaded successfully.Embedding dimension:{self.model.get_embedding_dimension()}")
    except Exception as e:
      print(f"Error loading model: {e}")
      raise
  def generate_embeddings(self, texts: List[str]) -> np.ndarray:
    """
    Generate embeddings for a list of texts

    Args:
        texts: List of text strings to embed

    Returns:
        numpy array of embeddings with shape (len(texts), embedding_dim)
    """

    if not self.model:
        raise ValueError("Model not loaded")

    print(f"Generating embeddings for {len(texts)} texts...")
    
    embeddings = self.model.encode(
        texts,
        show_progress_bar=True
    )

    print(f"Generated embeddings with shape: {embeddings.shape}")
    
    return embeddings
  ##initialize embedding manager
embedding_manager = EmbeddingManager()
EmbeddingManager

Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10533.57it/s]


Model loaded successfully.Embedding dimension:384


__main__.EmbeddingManager

### VectorStore

In [ ]:
class VectorStore:
  def __init__(
    self,
    collection_name: str = "my_collection",
    chroma_db_path: str = "./chroma_db"):
    self.collection_name = collection_name
    self.chroma_db_path = chroma_db_path
    self.client = None
    self.collection = None
    self._initialize_chromadb()
  def _initialize_chromadb(self):
    """Initialize the ChromaDB client and collection."""
    try:
      print(f"Initializing ChromaDB at path: {self.chroma_db_path}...")
      self.client = chromadb.Client(Settings(chroma_db_impl="duckdb+parquet", persist_directory=self.chroma_db_path))
      if self.collection_name in self.client.list_collections():
        print(f"Collection '{self.collection_name}' already exists. Loading existing collection.")
        self.collection = self.client.get_collection(name=self.collection_name)
      else:
        print(f"Creating new collection: {self.collection_name}")
        self.collection = self.client.create_collection(name=self.collection_name)
      print("ChromaDB initialized successfully.")
    except Exception as e:
      print(f"Error initializing ChromaDB: {e}")
      raise